# Module 5 — Inverse Kinematics
## Unit 8 — Mini Project: Reach the Fruit
### Lesson 8.2 — Building the Solver (Analytical + Numerical)

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
solve(): analytical branch, and numerical branch seeded to recover both elbows.


In [ ]:
import numpy as np

def fk_two_link(t1, t2, L1, L2):
    return np.array([L1*np.cos(t1)+L2*np.cos(t1+t2), L1*np.sin(t1)+L2*np.sin(t1+t2)])

def jacobian_2link(t1, t2, L1, L2):
    s1,s12=np.sin(t1),np.sin(t1+t2); c1,c12=np.cos(t1),np.cos(t1+t2)
    return np.array([[-L1*s1-L2*s12,-L2*s12],[L1*c1+L2*c12,L2*c12]])

def ik_2link_closed(x, y, L1, L2):
    c2=(x*x+y*y-L1*L1-L2*L2)/(2*L1*L2)
    if c2<-1-1e-9 or c2>1+1e-9: return []
    c2=max(-1.0,min(1.0,c2)); sols=[]
    for sign in (+1.0,-1.0):
        s2=sign*np.sqrt(max(0.0,1-c2*c2)); t2=np.arctan2(s2,c2)
        t1=np.arctan2(y,x)-np.arctan2(L2*np.sin(t2),L1+L2*np.cos(t2))
        sols.append((t1,t2))
        if abs(s2)<1e-6: break
    return sols

def reachable(x, y, L1, L2, tol=1e-9):
    r=np.hypot(x,y); return abs(L1-L2)-tol<=r<=L1+L2+tol

def ik_numerical(target, theta0, L1, L2, alpha=1.0, tol=1e-6, max_iter=100):
    theta=np.array(theta0,float); target=np.array(target,float)
    for it in range(max_iter):
        e=target-fk_two_link(theta[0],theta[1],L1,L2)
        if np.linalg.norm(e)<tol: return theta
        J=jacobian_2link(theta[0],theta[1],L1,L2)
        theta=theta+alpha*np.linalg.pinv(J)@e
    return theta

def norm180(a): return (a+180.0)%360.0-180.0

def solve(target, L1, L2, use_numerical=False, seeds=None):
    if not use_numerical:
        return ik_2link_closed(target[0],target[1],L1,L2)
    if seeds is None: seeds=[np.radians([10,20]), np.radians([10,-20])]
    out=[]
    for s in seeds:
        if reachable(target[0],target[1],L1,L2):
            sol=ik_numerical(target,s,L1,L2)
            if not any(np.allclose(sol,o,atol=1e-4) for o in out): out.append(tuple(sol))
    return out

L1,L2=0.4,0.3; target=(0.5,0.2)
ana=solve(target,L1,L2)
num=solve(target,L1,L2,use_numerical=True)
print("analytical:", [tuple(np.round(np.degrees(s),1)) for s in ana])
print("numerical :", [tuple(np.round(np.degrees(s),1)) for s in num])


## Coding Exercise (§8)
Branches agree; unreachable → []; two seeds recover both elbows.


In [ ]:
# YOUR CODE HERE
ana=solve(target,L1,L2); num=solve(target,L1,L2,use_numerical=True)
assert len(ana)==2 and len(num)==2
# every analytical solution is matched by a numerical one (FK same point)
for a in ana:
    assert any(np.allclose(fk_two_link(*a,L1,L2), fk_two_link(*n,L1,L2), atol=1e-4) for n in num)
assert solve((0.9,0.0),L1,L2)==[]                  # unreachable
signs=sorted(np.sign(s[1]) for s in num)
assert signs==[-1.0,1.0]                            # both elbows
print("solver core OK")


## Check your work


In [ ]:
assert len(solve((0.5,0.2),L1,L2))==2 and solve((0.95,0),L1,L2)==[]
print("All checks passed.")
